In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# c02 新目录重跑 01

本入口用于用户已明确授权的新增一次完整验证。原 fixed-c02-v1 已产出中断失败记录，继续保留。本次固定写入 fixed-c02-rerun01-v1，首次运行执行原 A/B 物化、时间接收和 DCT2＋CDC32 载荷读取；此新目录一旦开始即消费本次尝试，之后再次 Run all 只读本次保存结果。

SOURCE 已填好，选择 GPU 运行时后运行全部。复用现有整链入口及容量错误诊断，FP32、缓存位置、求解器、方法、固定分母和每次预算不变；本入口不表示容量故障已得到真实验证。
固定旧 CAL 玻璃工坊 c02 的 P0，一次执行原 A/B 物化、43 项时间接收、256 项载荷读取。time global_start=0；旧 a=11 仅管理/后评。不是全项目未见或全部组件独立留出。SOURCE、OUTPUT 已填写；普通源码位置可编辑，无摘要或版本门禁。默认远程入口从项目开发分支加载本次整包源码；也可填写已有源码目录。

全部 B/W/U 计划先冻结，再共享 60 个实际 E41。所有时间和载荷盲终态保存后才统一后评，读取原 SOURCE/protocol.json 的 config.writer.bits 与 c02 记录；不可读或非法不回退常数。旧算法、hard/L/submit、DCT2/CDC32、冲突擦除和 native 有限候选规则原样复用。

固定未来预算 A8457、B57、time43、payload256；异类算术上界8813不是模型推理次数。codec33（encode7/probe13/decode13）；post 时间1、载荷1、总汇总1单列，不增加观察。固定时间822节点/817边、载荷raw9600位/vote76800/local1440/final480，每臂160。各原阶段资源限制继续在实际路径执行，记录分阶段与总耗时，不把载荷1800秒套整包。

第一次真实调用前整包预约并保存纯数据恢复骨架；失败/中断连未进入阶段一起消费。再次 Run all 仅展示已存回执，不补物化/读取/评分、不打开位缓存或 truth。首轮记录保存失败时不后评。完整文件和 result 变量保留；默认只显示有界摘要。

五个 view 各 B32/32 且所有支持精确正确才达到 B160/160 目标，不能跨 view 拼凑。所有已提交时间位置/精确边包括旧 H 错误均保留；逐帧覆盖另报。B/U 共同位纠错、新输出支持、归属修正、丢失分别报告；B=U 同好可支持载荷迁移，但不能证明同步作用。仅归属修正不称解码纠错或 BER 下降。

失败摘要优先展示最早阶段错误、阶段状态和已有调用计数。磁盘容量比较保存 disk_cache_capacity.json（路径、free、channels、形状估算、minimum/required）；32GiB下限、FP32、原缓存位置和46块不变。旧结果缺少数值回执时不补测或猜测。确证未进入时间接收的822节点/817边只读解释为NOT_EXECUTED，缺失/未知回执仍为UNKNOWN，原evaluation不修改。

本包没有新增拒绝阈值或负例预算；W 错组不是 Ø。existence/null/FPR/隐藏错时间/来源认证未评估。原 native181/181 精确目标失败、4000步最优性及画质未证保留；旧失败不被新读回覆盖。本地交付仅做工程检查；新目录的真实整链结果由本次用户运行产生。


In [ ]:
SOURCE = '/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1'
OUTPUT = '/content/drive/MyDrive/Video-WM/B-New-Source-Chain-Confirmation-V1/fixed-c02-rerun01-v1'
SOURCE_DIRECTORY = None
SOURCE_REF = 'dev/local-joint-state-payload-v1'
MODEL_ID = None


In [ ]:
from pathlib import Path
import importlib.util, json, sys, tempfile, urllib.request
if (Path(OUTPUT)/'attempt.json').exists():
    result = {'status': 'READ_ONLY_EXISTING_PACKAGE', 'physical_calls_this_invocation': 0, 'post_calls_this_invocation': 0, 'saved': {}, 'read_errors': {}}
    for name in ('attempt', 'state', 'counts', 'resources', 'evaluation', 'result'):
        try:
            saved = json.loads((Path(OUTPUT)/(name+'.json')).read_text())
            if not isinstance(saved, dict): raise ValueError('saved receipt object required')
            result['saved'][name] = saved
        except (OSError, ValueError) as error: result['read_errors'][name] = str(error)
    # Record-only display uses the fixed skeleton saved before the first operation.
    # No source, planner, bit cache, truth, model or evaluator is loaded here.
    import copy
    result['recovered_artifacts'] = {}
    states = {}
    for item in result['saved'].get('attempt', {}).get('recovery_layout', []):
        value, error, unconfirmed = copy.deepcopy(item['fallback']), None, None
        state_path = item['state_path']
        if state_path not in states:
            try: states[state_path] = json.loads((Path(OUTPUT)/state_path).read_text())
            except (OSError, ValueError): states[state_path] = {}
        state = states[state_path] if isinstance(states[state_path], dict) else {}
        try:
            path = (Path(OUTPUT)/item['path']).resolve()
            if not path.is_relative_to(Path(OUTPUT).resolve()): raise ValueError('receipt path outside output')
            saved = json.loads(path.read_text())
            if item['kind'] == 'raw_bank':
                if not isinstance(saved, dict): raise ValueError('raw metadata object required')
                for key in value:
                    fixed, row = value[key], saved.get(key)
                    statuses = [state[op].get('status') if isinstance(state.get(op), dict) else None for op in fixed['operations']]
                    known = 'RETURNED' if statuses[-1] == 'RETURNED' else next((s for s in reversed(statuses) if s in ('FAILED', 'INTERRUPTED', 'RESOURCE_UNRESOLVED')), None)
                    if known is None: known = 'DEPENDENCY_UNRESOLVED' if all(s in ('NOT_STARTED', 'DEPENDENCY_UNRESOLVED') for s in statuses) else 'UNRESOLVED_RECEIPT'
                    valid = isinstance(row, dict) and known != 'UNRESOLVED_RECEIPT' and row.get('status') == known and all(row.get(k) == fixed[k] for k in ('raw_bit_slots', 'vote_slots', 'encoder_input_support'))
                    if valid: value[key] = dict(row, **{k: fixed[k] for k in ('raw_bit_slots', 'vote_slots', 'encoder_input_support', 'operations')})
                    else:
                        fixed.update(operation_status=known, unconfirmed_raw_metadata=row)
                        error = 'raw metadata incomplete or unconfirmed; fixed unit fields retained'
            elif not isinstance(state.get(item['operation']), dict) or state[item['operation']].get('status') != 'RETURNED':
                unconfirmed, error = saved, 'operation receipt not returned; saved artifact retained as unconfirmed'
            elif item['kind'] == 'time':
                if not isinstance(saved, dict): raise ValueError('time submission object required')
                for key, coordinate in (('final_nodes', 'received_frame'), ('final_relations', 'received_left')):
                    rows = saved.get(key)
                    if not isinstance(rows, list): raise ValueError('fixed time rows missing')
                    for r in range(len(value[key])):
                        if r < len(rows) and isinstance(rows[r], dict) and rows[r].get(coordinate) == r: value[key][r] = rows[r]
                        else: error = 'incomplete time receipt; fixed rows retained'
                    if len(rows) != len(value[key]): error = 'time row count mismatch'
            else:
                rows = saved.get('slots') if isinstance(saved, dict) else None
                if not isinstance(rows, list): raise ValueError('slot receipt list required')
                for i, row in enumerate(rows[:32]):
                    if isinstance(row, dict) and type(row.get('slot')) is int and row['slot'] == i and 'value' in row and (row['value'] is None or type(row['value']) is int and row['value'] in (0, 1)) and isinstance(row.get('status'), str): value['slots'][i] = row
                    else: error = 'invalid slot record retained as unresolved'
                if len(rows) != 32: error = 'incomplete slot receipt; fixed32 retained'
        except (OSError, ValueError, TypeError) as exc: error = str(exc)
        result['recovered_artifacts'][item['path']] = {'kind': item['kind'], 'value': value, 'error': error, 'unconfirmed_artifact': unconfirmed}
    if not result['recovered_artifacts']: result['read_errors']['fixed_slots'] = 'saved attempt skeleton unreadable; use the standalone read-only CLI to reconstruct fixed unresolved slots'
elif Path(OUTPUT).exists() and (not Path(OUTPUT).is_dir() or any(Path(OUTPUT).iterdir())):
    result = {'status': 'OUTPUT_CONFLICT', 'physical_calls_this_invocation': 0}
else:
    try:
        with tempfile.TemporaryDirectory(prefix='time-payload-bootstrap-') as temporary:
            root = Path(SOURCE_DIRECTORY) if SOURCE_DIRECTORY is not None else Path(temporary)
            if SOURCE_DIRECTORY is None:
                files = ('scripts/b_new_source_chain_confirmation_v1_entry.py', 'scripts/b_conditional_time_output_v1_entry.py', 'scripts/b_framewise_qim_v1_entry.py', 'experiments/wan_state_clock/b_new_source_chain_confirmation_v1_records.py', 'experiments/wan_state_clock/b_new_source_chain_confirmation_v1_diagnostics.py', 'experiments/wan_state_clock/b_minimal_l_v1_records.py', 'experiments/wan_state_clock/b_minimal_l_materialization_v1_records.py', 'experiments/wan_state_clock/b_framewise_qim_v1_records.py', 'experiments/wan_state_clock/b_framewise_qim_saved_candidate_v1_records.py', 'experiments/wan_state_clock/b_conditional_time_validation_v1_records.py', 'experiments/wan_state_clock/b_conditional_time_output_v1_records.py', 'experiments/wan_state_clock/b_time_payload_cdc32_v1_records.py', 'experiments/wan_state_clock/configs/b_new_source_chain_confirmation_v1.json')
                for relative in files:
                    path = root/relative; path.parent.mkdir(parents=True, exist_ok=True)
                    urllib.request.urlretrieve('https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/'+SOURCE_REF+'/'+relative, path)
            spec = importlib.util.spec_from_file_location('time_payload_entry', root/'scripts/b_new_source_chain_confirmation_v1_entry.py')
            entry = importlib.util.module_from_spec(spec)
            previous = sys.dont_write_bytecode; sys.dont_write_bytecode = True
            try: spec.loader.exec_module(entry)
            finally: sys.dont_write_bytecode = previous
            result = entry.launch(SOURCE, OUTPUT, SOURCE_DIRECTORY, SOURCE_REF, MODEL_ID)
    except Exception as error:
        result = {'status': 'PREPARATION_FAILED', 'error': str(error), 'physical_calls_this_invocation': 0}
# Bounded, receipt-only diagnostics also work for previously consumed outputs.
"""Bounded execution-receipt display only. No files, science, post or model calls.

The builder embeds this same stdlib-free code for source-free notebook recovery.
"""


def _receipt_object(value):
    return value if isinstance(value,dict) else {}


def operation_not_executed(row):
    row=_receipt_object(row)
    # Explicit operation state is evidence; a fabricated/missing fallback is not.
    if row.get('status') not in ('NOT_STARTED','DEPENDENCY_UNRESOLVED','NOT_EXECUTED'):return False
    return all(k not in row or type(row[k]) is int and row[k]==0 for k in ('started','returned'))


def _count_rows(value):
    value=_receipt_object(value)
    return _receipt_object(value.get('operations',value))


def execution_diagnostics(result,counts=None,stages=None,phase_rows=None):
    result=_receipt_object(result);stages=_receipt_object(stages);counts=_receipt_object(counts)
    chain=_receipt_object(_receipt_object(result.get('stage_results')).get('time_chain'))
    children=_receipt_object(chain.get('stage_results'))
    phases=_receipt_object(phase_rows) or _receipt_object(result.get('phases'))
    stage_results=dict(A=children.get('A'),B=children.get('B'),time=children.get('receiver'),
                       payload=_receipt_object(result.get('stage_results')).get('payload'))
    first=None;phase_summary={};calls={}
    for name in ('A','B','time','payload'):
        stored=_receipt_object(stages.get(name));stage=_receipt_object(stage_results[name]) or _receipt_object(stored.get('result'))
        phase=_receipt_object(phases.get(name))
        phase_summary[name]={k:phase.get(k) for k in ('status','started','returned')}
        phase_summary[name]['result_status']=stage.get('status',phase.get('result_status'))
        errors=_receipt_object(stage.get('errors'))
        if not errors:errors=_receipt_object(_receipt_object(chain.get('materialization_errors')).get(name))
        dependency={'B':'B_dependency','time':'receiver_dependency'}.get(name)
        if not errors and dependency in _receipt_object(chain.get('errors')):
            errors={'dependency':chain['errors'][dependency]}
        if not errors and phase.get('status') in ('FAILED','INTERRUPTED') and phase.get('error'):
            errors={'phase':phase['error']}
        if first is None and errors:
            key='stage' if 'stage' in errors else next(iter(errors))
            first=dict(stage=name,operation=str(key)[:80],error=str(errors[key])[:800],
                       phase_status=phase.get('status'),result_status=stage.get('status'))
        rows=_count_rows(counts.get(name)) or _count_rows(stored.get('counts')) or _count_rows(stage.get('counts'))
        calls[name]=dict(recorded_kinds=len(rows),started_nonzero={},unstarted=[],unknown=[])
        for key,value in list(rows.items())[:32]:
            row=_receipt_object(value);started=row.get('started');returned=row.get('returned');label=str(key)[:64]
            if type(started) is int and started>0:
                calls[name]['started_nonzero'][label]=dict(started=started,returned=returned if type(returned) is int else None)
            elif type(started) is int and started==0 and type(returned) is int and returned==0:
                calls[name]['unstarted'].append(label)
            else:calls[name]['unknown'].append(label)
    if first is None:
        errors=_receipt_object(result.get('errors'))
        if errors:
            key=next(iter(errors));first=dict(stage='package',operation=str(key)[:80],error=str(errors[key])[:800])
    if first and ('insufficient space for FP32 native feature cache' in first['error'] or 'disk cache capacity' in first['error']):
        first.update(failure_kind='DISK_CACHE_CAPACITY_CHECK',
                     interpretation='filesystem disk-cache check; not a CUDA or RAM allocation failure')
    for name in ('time_post','payload_post','summary'):
        phase=_receipt_object(phases.get(name))
        phase_summary[name]={k:phase.get(k) for k in ('status','started','returned','result_status')}
    phase=_receipt_object(phases.get('time'));rows=_count_rows(counts.get('time'))
    positive=any(any(type(r.get(k)) is int and r[k]>0 for k in ('started','returned'))
                 for r in rows.values() if isinstance(r,dict))
    unentered=(operation_not_executed(phase) and phase.get('started')==0 and not positive)
    receiver='NOT_EXECUTED' if unentered else 'EXECUTION_RECORDED' if phase.get('started')==1 or positive else 'UNKNOWN'
    zeros={k:0 for k in ('CORRECT','WRONG','UNRESOLVED','NOT_EXECUTED','UNKNOWN')}
    time_counts=dict(nodes=dict(zeros,NOT_EXECUTED=822),relations=dict(zeros,NOT_EXECUTED=817)) if unentered else None
    capacity=_receipt_object(_receipt_object(stages.get('A')).get('disk_cache_capacity'))
    return dict(earliest_stage_error=first,phase_states=phase_summary,operation_receipts=calls,
        receiver_execution=receiver,time_execution_counts=time_counts,
        time_counts_basis='READ_ONLY_EXECUTION_RECEIPTS_NOT_REEVALUATION' if unentered else None,
        disk_cache_capacity={k:capacity.get(k) for k in ('status','check_path','actual_free_bytes','channels',
            'shape_estimate_bytes','minimum_bytes','required_bytes')} if capacity else None,
        missing_capacity_measurements='not recorded; historical free/channels/required cannot be reconstructed' if not capacity else None)


def display_summary(result,output,saved=None):
    result=_receipt_object(result);saved=_receipt_object(saved) or _receipt_object(result.get('saved'))
    actual=_receipt_object(saved.get('result')) or result
    evaluation=_receipt_object(saved.get('evaluation'));transfer=_receipt_object(evaluation.get('payload_transfer'))
    diagnostic=_receipt_object(result.get('execution_diagnostics')) or _receipt_object(actual.get('execution_diagnostics')) or execution_diagnostics(actual,saved.get('counts'),
        result.get('stages'),saved.get('state'))
    times=diagnostic.get('time_execution_counts') or _receipt_object(evaluation.get('time')).get('counts')
    errors=_receipt_object(actual.get('errors'))
    return dict(status=result.get('status'),output=str(output),evaluation_status=actual.get('evaluation_status',evaluation.get('status')),
        earliest_stage_error=diagnostic['earliest_stage_error'],phase_states=diagnostic['phase_states'],
        operation_receipts=diagnostic['operation_receipts'],disk_cache_capacity=diagnostic['disk_cache_capacity'],
        missing_capacity_measurements=diagnostic['missing_capacity_measurements'],receiver_execution=diagnostic['receiver_execution'],
        time_counts=times,time_counts_basis=diagnostic['time_counts_basis'] or 'SAVED_EVALUATION',
        B_correct_supported=transfer.get('correct_supported'),B_fixed_slots=160,five_view_target=transfer.get('full_five_view_target_observed'),
        synchronization_effect=_receipt_object(evaluation.get('synchronization_effect')).get('status'),
        physical_calls_this_invocation=result.get('physical_calls_this_invocation'),post_calls_this_invocation=result.get('post_calls_this_invocation'),
        fixed_denominators=dict(time_nodes=822,time_edges=817,raw_bits=9600,votes=76800,local=1440,final=480),
        arithmetic_upper_bound=8813,not_model_inference_count=True,codec=dict(encode=7,probe=13,decode=13),
        total_wall_seconds=_receipt_object(saved.get('resources')).get('total_wall_seconds',actual.get('wall_seconds')),
        recovered_records=len(_receipt_object(result.get('recovered_artifacts'))),
        downstream_errors={str(k)[:80]:str(v)[:240] for k,v in list(errors.items())[:8]},
        full_records='Complete saved records remain unchanged in OUTPUT; display does not rerun post.')

saved = result.get('saved', {})
for name in ('evaluation', 'result', 'resources', 'counts', 'state'):
    if name not in saved:
        try:
            value = json.loads((Path(OUTPUT)/(name+'.json')).read_text())
            if isinstance(value, dict): saved[name] = value
        except (OSError, ValueError): pass
stages = result.setdefault('stages', {})
for stage, folder in (('A', 'time_chain/materialization/A'), ('B', 'time_chain/materialization/B'), ('time', 'time_chain/receiver'), ('payload', 'payload')):
    receipts = stages.setdefault(stage, {})
    for name in ('result', 'counts', 'disk_cache_capacity'):
        if name not in receipts:
            try:
                value = json.loads((Path(OUTPUT)/folder/(name+'.json')).read_text())
                if isinstance(value, dict): receipts[name] = value
            except (OSError, ValueError): pass
summary = display_summary(result, OUTPUT, saved)
print(summary)
